In [0]:
project_identifier = 'dac023'

# IG thresholds: a column is EXCLUDED only if it has a tag whose value exceeds the threshold.
# Untagged columns are INCLUDED (treated as within thresholds).
max_ig_risk = 4
max_ig_severity = 4

# Always exclude these columns regardless of tags.
columns_to_exclude = ['ADC_UPDT']

# Always include these columns regardless of tags.
# Format: {'catalog.schema.table': ['col1', 'col2']} or use '*' as table key for all tables.
columns_to_include = {}

# Tables that should bypass IG filtering entirely (SELECT * with no column filtering).
# Still cohort-filtered if a person_id column exists.
unfiltered_tables = []

# Source for NHS_Number / MRN enrichment. Every view with a resolvable person_id column
# gets these two columns appended via LEFT JOIN, so identifiers are consistent across views.
demographics_source = '4_prod.rde.rde_patient_demographics'

# ---------------------------------------------------------------------------
# Table lists
# ---------------------------------------------------------------------------

# RDE tables (from 4_prod.rde) — DAC023's original list
rde_tables = [
    'rde_aliases', 'rde_all_procedures', 'rde_all_diagnosis', 'rde_allergydetails',
    'rde_apc_diagnosis', 'rde_apc_opcs', 'rde_ariapharmacy', 'rde_blobdataset',
    'rde_cds_apc', 'rde_cds_opa', 'rde_critactivity', 'rde_critopcs', 'rde_critperiod',
    'rde_emergencyd', 'rde_emergency_findings', 'rde_encounter', 'rde_family_history',
    'rde_iqemo', 'rde_mat_nnu_episodes', 'rde_mat_nnu_exam', 'rde_mat_nnu_nccmds',
    'rde_measurements', 'rde_medadmin', 'rde_mill_powertrials', 'rde_msds_booking',
    'rde_msds_carecontact', 'rde_msds_delivery', 'rde_msds_diagnosis',
    'rde_op_diagnosis', 'rde_opa_opcs', 'rde_pathology', 'rde_patient_demographics',
    'rde_pc_diagnosis', 'rde_pc_problems', 'rde_pc_procedures', 'rde_pharmacyorders',
    'rde_powerforms', 'rde_radiology', 'rde_raw_pathology', 'rde_scr_careplan',
    'rde_scr_deftreatment', 'rde_scr_demographics', 'rde_scr_diagnosis',
    'rde_scr_investigations', 'rde_scr_pathology', 'rde_scr_referrals',
    'rde_scr_trackingcomments',
]

# Bronze/map tables (from 4_prod.bronze)
# NOTE: map_address is created as a bespoke view below because it links to the cohort
# via PARENT_ENTITY_ID + PARENT_ENTITY_NAME='PERSON', not a standard person_id column.
map_tables = []

# OMOP tables (from 4_prod.omop)
omop_tables = []

# Raw/mill tables (from 4_prod.raw)
mill_tables = []

# Pharos-specific silver tables (from the Pharos pipeline in 4_prod.silver).
# Project-curated, so SELECT * (no IG filtering) but still cohort-filtered.
pharos_tables = [
    'pharos_person', 'pharos_medical_history', 'pharos_tumour',
    'pharos_imaging', 'pharos_pathology', 'pharos_treatment',
    'pharos_followup', 'pharos_comorbidities',
]

## Cohort

In [0]:
cohort_sql = f"""
CREATE OR REPLACE VIEW 6_mgmt.cohorts.{project_identifier} AS
WITH matched_aliases AS (
    SELECT DISTINCT
        l.NHS_NUMBER,
        TRY_CAST(pa.PERSON_ID AS BIGINT) AS PERSON_ID
    FROM 6_mgmt.cohort_lookup.phar001_lookup l
    JOIN 4_prod.raw.mill_person_alias pa
        ON REGEXP_REPLACE(l.NHS_NUMBER, '[ -]', '') = REGEXP_REPLACE(pa.ALIAS, '[ -]', '')
)
SELECT DISTINCT
    PERSON_ID,
    CAST(NULL AS STRING) AS SUBCOHORT
FROM matched_aliases
WHERE PERSON_ID IS NOT NULL
"""
spark.sql(cohort_sql)
print(f"Created cohort view: 6_mgmt.cohorts.{project_identifier}")

## Schema Setup

In [0]:
spark.sql("USE CATALOG 5_projects")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS 5_projects.{project_identifier}")

existing_views_df = spark.sql(f"SHOW VIEWS IN 5_projects.{project_identifier}")
existing_views_set = set()
if existing_views_df.count() > 0:
    existing_views_set = {row.viewName for row in existing_views_df.collect()}
    for view_name in existing_views_set:
        spark.sql(f"DROP VIEW IF EXISTS 5_projects.{project_identifier}.{view_name}")
        print(f"Dropped view: 5_projects.{project_identifier}.{view_name}")

# Drop any stray tables (e.g. materialised results from previous runs)
existing_tables_df = spark.sql(f"SHOW TABLES IN 5_projects.{project_identifier}")
for row in existing_tables_df.collect():
    if row.tableName not in existing_views_set and row.tableName != project_identifier:
        spark.sql(f"DROP TABLE IF EXISTS 5_projects.{project_identifier}.{row.tableName}")
        print(f"Dropped table: 5_projects.{project_identifier}.{row.tableName}")

## Helper Functions

In [0]:
def get_safe_columns(catalog, schema, table):
    """
    Return a list of column names that pass IG filtering for the given table.

    A column is EXCLUDED if:
      - It has an ig_risk tag with value > max_ig_risk, OR
      - It has an ig_severity tag with value > max_ig_severity, OR
      - It appears in columns_to_exclude.

    Untagged columns are INCLUDED (treated as within thresholds).
    columns_to_include can override tag-based exclusions (but not columns_to_exclude).
    """
    full_path = f"{catalog}.{schema}.{table}"
    all_columns = spark.table(full_path).columns

    over_threshold_df = spark.sql(f"""
        SELECT DISTINCT column_name
        FROM {catalog}.information_schema.column_tags
        WHERE schema_name = '{schema}'
          AND table_name = '{table}'
          AND (
              (tag_name = 'ig_risk'     AND CAST(tag_value AS INT) > {max_ig_risk})
           OR (tag_name = 'ig_severity' AND CAST(tag_value AS INT) > {max_ig_severity})
          )
    """)
    over_threshold = set(over_threshold_df.toPandas()['column_name'].tolist())

    includes = set(columns_to_include.get(full_path, []))
    includes |= set(columns_to_include.get('*', []))

    safe = (set(all_columns) - over_threshold) | includes
    safe -= set(columns_to_exclude)

    return [c for c in all_columns if c in safe]


def find_person_id_column(catalog, schema, table):
    """Find the person ID column in a table, checking common name variations."""
    full_path = f"{catalog}.{schema}.{table}"
    columns = spark.table(full_path).columns

    for candidate in ['PERSON_ID', 'person_id', 'Person_ID', 'PERSONID', 'PersonID', 'participant_id']:
        if candidate in columns:
            return candidate

    for col in columns:
        if 'person' in col.lower() and 'id' in col.lower():
            return col

    return None


def create_cohort_filtered_view(catalog, schema, table, project_id, column_list=None, alias=None):
    """
    Create a cohort-filtered view in 5_projects.{project_id}.

    NHS_Number and MRN are appended via LEFT JOIN to {demographics_source}
    whenever the source table has a resolvable person_id column, so identifiers
    are consistent across all project views.

    Args:
        catalog: Source catalog (e.g. '4_prod')
        schema: Source schema (e.g. 'rde', 'bronze', 'omop', 'silver')
        table: Source table name
        project_id: Project identifier for the target schema and cohort
        column_list: Explicit columns to select. If None, uses get_safe_columns().
                     Pass ['*'] to select all columns (for unfiltered reference tables).
        alias: Optional view name override. Defaults to the source table name.
    """
    full_path = f"{catalog}.{schema}.{table}"
    view_name = alias or table

    source_columns = spark.table(full_path).columns
    source_cols_lower = {c.lower() for c in source_columns}

    if column_list is None:
        column_list = get_safe_columns(catalog, schema, table)
        if not column_list:
            print(f"WARNING: No columns passed IG filter for {full_path}. Skipping.")
            return False

    person_id_col = find_person_id_column(catalog, schema, table)

    if column_list == ['*']:
        columns_sql = "t.*"
    else:
        columns_sql = ", ".join([f"t.`{c}`" for c in column_list])

    # Append NHS_Number / MRN where possible. Skip:
    #   - rde_patient_demographics itself (it's the source — would duplicate columns)
    #   - tables with no person_id column (no join key)
    #   - columns already present on the source (would create duplicate column names)
    is_demographics_source = (f"{catalog}.{schema}.{table}" == demographics_source)
    extras = []
    if person_id_col and not is_demographics_source:
        if 'nhs_number' not in source_cols_lower:
            extras.append('d.NHS_Number')
        if 'mrn' not in source_cols_lower:
            extras.append('d.MRN')

    if extras:
        select_clause = f"{columns_sql}, {', '.join(extras)}"
        demog_join = f"LEFT JOIN {demographics_source} d ON t.{person_id_col} = d.PERSON_ID"
    else:
        select_clause = columns_sql
        demog_join = ""

    if person_id_col:
        view_sql = f"""
        CREATE OR REPLACE VIEW 5_projects.{project_id}.{view_name} AS
        SELECT {select_clause}
        FROM {full_path} t
        INNER JOIN 6_mgmt.cohorts.{project_id} c
            ON t.{person_id_col} = c.PERSON_ID
        {demog_join}
        """
    else:
        print(f"Note: No person ID column in {full_path}. Creating view without cohort filtering or NHS/MRN enrichment.")
        view_sql = f"""
        CREATE OR REPLACE VIEW 5_projects.{project_id}.{view_name} AS
        SELECT {columns_sql}
        FROM {full_path} t
        """

    spark.sql(view_sql)
    cols_label = len(column_list) if column_list != ['*'] else 'all'
    enrich_note = f" + {len(extras)} demographics col(s)" if extras else ""
    print(f"Created view: 5_projects.{project_id}.{view_name} ({cols_label} columns{enrich_note})")
    return True

## Create Views

In [0]:
created = []
failed = []

# --- RDE tables ---
for table in rde_tables:
    try:
        if create_cohort_filtered_view('4_prod', 'rde', table, project_identifier):
            created.append(f"rde.{table}")
    except Exception as e:
        failed.append((f"rde.{table}", str(e)))
        print(f"ERROR: rde.{table}: {e}")

# --- Bronze/map tables ---
for table in map_tables:
    try:
        if create_cohort_filtered_view('4_prod', 'bronze', table, project_identifier):
            created.append(f"bronze.{table}")
    except Exception as e:
        failed.append((f"bronze.{table}", str(e)))
        print(f"ERROR: bronze.{table}: {e}")

# --- OMOP tables ---
for table in omop_tables:
    try:
        if create_cohort_filtered_view('4_prod', 'omop', table, project_identifier):
            created.append(f"omop.{table}")
    except Exception as e:
        failed.append((f"omop.{table}", str(e)))
        print(f"ERROR: omop.{table}: {e}")

# --- Raw/mill tables ---
for table in mill_tables:
    try:
        if create_cohort_filtered_view('4_prod', 'raw', table, project_identifier):
            created.append(f"raw.{table}")
    except Exception as e:
        failed.append((f"raw.{table}", str(e)))
        print(f"ERROR: raw.{table}: {e}")

# --- Pharos silver tables (project-curated, SELECT * with no IG filtering) ---
for table in pharos_tables:
    try:
        if create_cohort_filtered_view('4_prod', 'silver', table, project_identifier, column_list=['*']):
            created.append(f"silver.{table}")
    except Exception as e:
        failed.append((f"silver.{table}", str(e)))
        print(f"ERROR: silver.{table}: {e}")

# --- Unfiltered reference tables ---
for table in unfiltered_tables:
    try:
        if '.' in table:
            cat, sch, tbl = table.split('.')
        else:
            cat, sch, tbl = '4_prod', 'omop', table
        if create_cohort_filtered_view(cat, sch, tbl, project_identifier, column_list=['*']):
            created.append(f"{sch}.{tbl}")
    except Exception as e:
        failed.append((f"{table}", str(e)))
        print(f"ERROR: {table}: {e}")

## Bespoke Views

In [0]:
# map_address: cohort link is PARENT_ENTITY_ID where PARENT_ENTITY_NAME='PERSON',
# so the generic person_id detection doesn't apply.
try:
    addr_cols = [c for c in get_safe_columns('4_prod', 'bronze', 'map_address')
                 if c != 'PARENT_ENTITY_ID']
    addr_cols_sql = ", ".join([f"m.`{c}`" for c in addr_cols])
    spark.sql(f"""
        CREATE OR REPLACE VIEW 5_projects.{project_identifier}.map_address AS
        SELECT m.PARENT_ENTITY_ID AS PERSON_ID, {addr_cols_sql}, d.NHS_Number, d.MRN
        FROM 4_prod.bronze.map_address m
        INNER JOIN 6_mgmt.cohorts.{project_identifier} c
            ON m.PARENT_ENTITY_ID = c.PERSON_ID
        LEFT JOIN {demographics_source} d
            ON m.PARENT_ENTITY_ID = d.PERSON_ID
        WHERE m.PARENT_ENTITY_NAME = 'PERSON'
    """)
    print(f"Created view: 5_projects.{project_identifier}.map_address ({len(addr_cols) + 3} columns)")
    created.append("bronze.map_address")
except Exception as e:
    failed.append(("bronze.map_address", str(e)))
    print(f"ERROR: bronze.map_address: {e}")

## Schema Documentation View

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW 5_projects.{project_identifier}.schema AS
SELECT
    table_name,
    column_name,
    COALESCE(comment, '') AS column_comment
FROM 5_projects.information_schema.columns
WHERE table_catalog = '5_projects'
  AND table_schema = '{project_identifier}'
  AND table_name != 'schema'
ORDER BY table_name, column_name
""")
print(f"Created schema view: 5_projects.{project_identifier}.schema")

## Summary

In [0]:
print("=" * 60)
print(f"Project Setup Complete: {project_identifier}")
print("=" * 60)
print(f"\nCohort:  6_mgmt.cohorts.{project_identifier}")
print(f"Schema:  5_projects.{project_identifier}")
print(f"\nIG Thresholds: ig_risk <= {max_ig_risk}, ig_severity <= {max_ig_severity}")
print(f"Policy: untagged columns are INCLUDED (treated as within thresholds)")
print(f"Identifier enrichment: NHS_Number, MRN appended from {demographics_source}")
print(f"\nViews created: {len(created)}")
for v in created:
    print(f"  + {v}")
if failed:
    print(f"\nFailed: {len(failed)}")
    for t, e in failed:
        print(f"  x {t}: {e}")